<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# Sequential Recommender Quick Start

### Example: SLi_Rec : Adaptive User Modeling with Long and Short-Term Preferences for Personailzed Recommendation
Unlike a general recommender such as Matrix Factorization or xDeepFM (in the repo) which doesn't consider the order of the user's activities, sequential recommender systems take the sequence of the user behaviors as context and the goal is to predict the items that the user will interact in a short time (in an extreme case, the item that the user will interact next).

This notebook aims to give you a quick example of how to train a sequential model based on a public Amazon dataset. Currently, we can support NextItNet \[4\], GRU \[2\], Caser \[3\], A2SVD \[1\], SLi_Rec \[1\], and SUM \[5\]. Without loss of generality, this notebook takes [SLi_Rec model](https://www.microsoft.com/en-us/research/uploads/prod/2019/07/IJCAI19-ready_v1.pdf) for example.
SLi_Rec \[1\] is a deep learning-based model aims at capturing both long and short-term user preferences for precise recommender systems. To summarize, SLi_Rec has the following key properties:

* It adopts the attentive "Asymmetric-SVD" paradigm for long-term modeling;
* It takes both time irregularity and semantic irregularity into consideration by modifying the gating logic in LSTM.
* It uses an attention mechanism to dynamic fuse the long-term component and short-term component.

In this notebook, we test SLi_Rec on a subset of the public dataset: [Amazon_reviews](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Movies_and_TV_5.json.gz) and [Amazon_metadata](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_Movies_and_TV.json.gz)

This notebook is tested under TF 2.6. 

## 0. Global Settings and Imports

In [1]:
import os
import sys

import tensorflow.compat.v1 as tf

tf.get_logger().setLevel("ERROR")  # only show error messages

from recommenders.utils.timer import Timer
from recommenders.utils.constants import SEED
from recommenders.models.deeprec.deeprec_utils import prepare_hparams
from MG_amazon_reviews import data_preprocessing, _create_vocab, download_and_extract

from recommenders.models.deeprec.models.sequential.sli_rec import (
    SLI_RECModel as SeqModel,
)

####  to use the other model, use one of the following lines:
# from recommenders.models.deeprec.models.sequential.asvd import A2SVDModel as SeqModel
# from recommenders.models.deeprec.models.sequential.caser import CaserModel as SeqModel
# from recommenders.models.deeprec.models.sequential.gru import GRUModel as SeqModel
# from recommenders.models.deeprec.models.sequential.sum import SUMModel as SeqModel
# from recommenders.models.deeprec.models.sequential.nextitnet import NextItNetModel
from recommenders.models.deeprec.io.sequential_iterator import SequentialIterator

# from recommenders.models.deeprec.io.nextitnet_iterator import NextItNetIterator
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"Tensorflow version: {tf.__version__}")

System version: 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:20:01) [Clang 18.1.8 ]
Tensorflow version: 2.15.1


#### Parameters

In [ ]:
EPOCHS = 10
BATCH_SIZE = 400
RANDOM_SEED = SEED  # Set None for non-deterministic result

data_path = "../Data/domain_transfer_capabilities/"

##  ATTENTION: change to the corresponding config file, e.g., caser.yaml for CaserModel, sum.yaml for SUMModel
yaml_file = "../Experiments/domain_transfer_capabilities/recommenders/models/deeprec/config/sli_rec.yaml"

##  1. Input data format
The input data contains 8 columns, i.e.,   `<label> <user_id> <item_id> <category_id> <timestamp> <history_item_ids> <history_cateory_ids> <hitory_timestamp>`  columns are seperated by `"\t"`.  item_id and category_id denote the target item and category, which means that for this instance, we want to guess whether user user_id will interact with item_id at timestamp. `<history_*>` columns record the user behavior list up to `<timestamp>`, elements are separated by commas.  `<label>` is a binary value with 1 for positive instances and 0 for negative instances.  One example for an instance is: 

`1       A1QQ86H5M2LVW2  B0059XTU1S      Movies  1377561600      B002ZG97WE,B004IK30PA,B000BNX3AU,B0017ANB08,B005LAIHW2  Movies,Movies,Movies,Movies,Movies   1304294400,1304812800,1315785600,1316304000,1356998400` 

In data preprocessing stage, we have a script to generate some ID mapping dictionaries, so user_id, item_id and category_id will be mapped into interager index starting from 1. And you need to tell the input iterator where is the ID mapping files are. (For example, in the next section, we have some mapping files like user_vocab, item_vocab, and cate_vocab).  The data preprocessing script is at [recommenders/dataset/amazon_reviews.py](../../recommenders/dataset/amazon_reviews.py), you need to call the `_create_vocab(train_file, user_vocab, item_vocab, cate_vocab)` function. Note that ID vocabulary only creates from the train_file, so the new IDs in valid_file or test_file will be regarded as unknown IDs and assigned with a defualt 0 index.

Only the SLi_Rec model is time-aware. For the other models, you can just pad some meaningless timestamp in the data files to fill up the format, the models will ignore these columns.

We use Softmax to the loss function. In training and evalution stage, we group 1 positive instance with `num_ngs` negative instances. Pair-wise ranking can be regarded as a special case of softmax ranking, where `num_ngs` is set to 1. 

More specifically, for training and evalation, you need to organize the data file such that each one positive instance is followed by `num_ngs` negative instances. Our program will take `1+num_ngs` lines as a unit for Softmax calculation. `num_ngs` is a parameter you need to pass to the `prepare_hparams`, `fit` and `run_eval` function. `train_num_ngs` in `prepare_hparams` denotes the number of negative instances for training, where a recommended number is 4. `valid_num_ngs` and `num_ngs` in `fit` and `run_eval` denote the number in evalution. In evaluation, the model calculates metrics among the `1+num_ngs` instances. For the `predict` function, since we only need to calcuate a score for each individual instance, there is no need for `num_ngs` setting.  More details and examples will be provided in the following sections.

For training stage, if you don't want to prepare negative instances, you can just provide positive instances and set the parameter `need_sample=True, train_num_ngs=train_num_ngs` for function `prepare_hparams`, our model will dynamicly sample `train_num_ngs` instances as negative samples in each mini batch.

###  Amazon dataset
Now let's start with a public dataset containing product reviews and metadata from Amazon, which is widely used as a benchmark dataset in recommemdation systems field.

In [3]:
# for test
train_file = os.path.join(data_path, r"baby_train_data")
valid_file = os.path.join(data_path, r"baby_valid_data")
test_file = os.path.join(data_path, r"baby_test_data")
user_vocab = os.path.join(data_path, r"baby_user_vocab.pkl")
item_vocab = os.path.join(data_path, r"baby_item_vocab.pkl")
cate_vocab = os.path.join(data_path, r"baby_category_vocab.pkl")
output_file = os.path.join(data_path, r"baby_output.txt")

reviews_name = "Baby_Products.jsonl"
meta_name = "meta_Baby_Products.jsonl"
reviews_file = os.path.join(data_path, reviews_name)
meta_file = os.path.join(data_path, meta_name)
train_num_ngs = 4  # number of negative instances with a positive instance for training
valid_num_ngs = (
    4  # number of negative instances with a positive instance for validation
)
test_num_ngs = 9  # number of negative instances with a positive instance for testing
sample_rate = 0.01
# sample a small item set for training and testing here for fast example


input_files = [
    reviews_file,
    meta_file,
    train_file,
    valid_file,
    test_file,
    user_vocab,
    item_vocab,
    cate_vocab,
]

# Run data_preprocessing if train_file doesn't exist
if not os.path.exists(train_file):
    download_and_extract(reviews_name, reviews_file)
    download_and_extract(meta_name, meta_file)
    data_preprocessing(
        *input_files,
        sample_rate=sample_rate,
        valid_num_ngs=valid_num_ngs,
        test_num_ngs=test_num_ngs
    )

# Now, after data_preprocessing, create the vocab files
from MG_amazon_reviews import _create_vocab

In [4]:
_create_vocab(train_file, user_vocab, item_vocab, cate_vocab)

#### 1.1 Prepare hyper-parameters
prepare_hparams() will create a full set of hyper-parameters for model training, such as learning rate, feature number, and dropout ratio. We can put those parameters in a yaml file (a complete list of parameters can be found under our config folder) , or pass parameters as the function's parameters (which will overwrite yaml settings).

Parameters hints: <br>
`need_sample` controls whether to perform dynamic negative sampling in mini-batch. 
`train_num_ngs` indicates how many negative instances followed by one positive instances.  <br>
Examples: <br>
(1) `need_sample=True and train_num_ngs=4`:  There are only positive instances in your training file. Our model will dynamically sample 4 negative instances for each positive instances in mini-batch. Note that if need_sample is set to True, train_num_ngs should be greater than zero. <br>
(2) `need_sample=False and train_num_ngs=4`: In your training file, each one positive line is followed by 4 negative lines. Note that if need_sample is set to False, you must provide a traiing file with negative instances, and train_num_ngs should match the number of negative number in your training file.

In [5]:
### NOTE:
### remember to use `_create_vocab(train_file, user_vocab, item_vocab, cate_vocab)` to generate the user_vocab, item_vocab and cate_vocab files, if you are using your own dataset rather than using our demo Amazon dataset.
hparams = prepare_hparams(
    yaml_file,
    embed_l2=0.0,
    layer_l2=0.0,
    learning_rate=0.001,  # set to 0.01 if batch normalization is disable
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    show_step=20,
    MODEL_DIR=os.path.join(data_path, "baby_model_01/"),
    SUMMARIES_DIR=os.path.join(data_path, "baby_summary_01/"),
    user_vocab=user_vocab,
    item_vocab=item_vocab,
    cate_vocab=cate_vocab,
    need_sample=True,
    train_num_ngs=train_num_ngs,  # provides the number of negative instances for each positive instance for loss computation.
)

#### 1.2 Create data loader
Designate a data iterator for the model. All our sequential models use SequentialIterator. 
data format is introduced aboved. 

<br>Validation and testing data are files after negative sampling offline with the number of `<num_ngs>` and `<test_num_ngs>`.

In [6]:
input_creator = SequentialIterator
#### uncomment this for the NextItNet model, because it needs a special data iterator for training
# input_creator = NextItNetIterator

## 2. Create model
When both hyper-parameters and data iterator are ready, we can create a model:

In [7]:
model = SeqModel(hparams, input_creator, seed=RANDOM_SEED)

## sometimes we don't want to train a model from scratch
## then we can load a pre-trained model like this:
# model.load_model(r'your_model_path')

/opt/miniconda3/envs/recom/lib/python3.10/site-packages/recommenders/models/deeprec/models/base_model.py:701: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.BatchNormalization` instead. In particular, `tf.control_dependencies(tf.GraphKeys.UPDATE_OPS)` should not be used (consult the `tf.keras.layers.BatchNormalization` documentation).
  curr_hidden_nn_layer = tf.compat.v1.layers.batch_normalization(
2025-04-27 07:00:00.675781: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled


Now let's see what is the model's performance at this point (without starting training):

In [8]:
# test_num_ngs is the number of negative lines after each positive line in your test_file
print(model.run_eval(test_file, num_ngs=test_num_ngs))

{'auc': 0.5001, 'logloss': 0.6931, 'mean_mrr': 0.2736, 'ndcg@2': 0.1433, 'ndcg@4': 0.2293, 'ndcg@6': 0.3002, 'group_auc': 0.5003}


AUC=0.5 is a state of random guess. We can see that before training, the model behaves like random guessing.

#### 2.1 Train model
Next we want to train the model on a training set, and check the performance on a validation dataset. Training the model is as simple as a function call:

In [9]:
with Timer() as train_time:
    model = model.fit(train_file, valid_file, valid_num_ngs=valid_num_ngs)

# valid_num_ngs is the number of negative lines after each positive line in your valid_file
# we will evaluate the performance of model on valid_file every epoch
print("Time cost for training is {0:.2f} mins".format(train_time.interval / 60.0))

step 20 , total_loss: 1.6098, data_loss: 1.6098
step 40 , total_loss: 1.6095, data_loss: 1.6095
step 60 , total_loss: 1.6085, data_loss: 1.6085
step 80 , total_loss: 1.6090, data_loss: 1.6090
step 100 , total_loss: 1.6106, data_loss: 1.6106
eval valid at epoch 1: auc:0.5186,logloss:0.6905,mean_mrr:0.4669,ndcg@2:0.3421,ndcg@4:0.5267,ndcg@6:0.5977,group_auc:0.5173
step 20 , total_loss: 1.6007, data_loss: 1.6007
step 40 , total_loss: 1.5958, data_loss: 1.5958
step 60 , total_loss: 1.5612, data_loss: 1.5612
step 80 , total_loss: 1.5526, data_loss: 1.5526
step 100 , total_loss: 1.5236, data_loss: 1.5236
eval valid at epoch 2: auc:0.6004,logloss:0.6595,mean_mrr:0.5337,ndcg@2:0.4338,ndcg@4:0.5982,ndcg@6:0.6488,group_auc:0.5965
step 20 , total_loss: 1.4971, data_loss: 1.4971
step 40 , total_loss: 1.5134, data_loss: 1.5134
step 60 , total_loss: 1.5258, data_loss: 1.5258
step 80 , total_loss: 1.5066, data_loss: 1.5066
step 100 , total_loss: 1.4724, data_loss: 1.4724
eval valid at epoch 3: auc:0.

#### 2.2  Evaluate model

Again, let's see what is the model's performance now (after training):

In [10]:
res_syn = model.run_eval(test_file, num_ngs=test_num_ngs)
print(res_syn)

{'auc': 0.6834, 'logloss': 0.6599, 'mean_mrr': 0.4491, 'ndcg@2': 0.3468, 'ndcg@4': 0.4542, 'ndcg@6': 0.5159, 'group_auc': 0.6761}
